# 🔍 Standalone Single-Image Deepfake Predictor (Interactive Testing Tool)

This interactive playground uses the fine-tuned **Meta DINOv3 ViT-Small/16** model (`best_model_v3.pt`, 21.60M parameters) to classify any single user-supplied human face image as **REAL** (Pristine) or **FAKE** (Deepfake / Generative Synthesis).


### 1. Environment Initialization & Path Configuration


In [ ]:
import os, sys
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# Dynamic project root resolution
ROOT_DIR = Path(os.getcwd()).resolve()
if not (ROOT_DIR / "src").exists() and (ROOT_DIR.parent / "src").exists():
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Execution Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  • GPU: {torch.cuda.get_device_name(0)}")

# Checkpoint path
CKPT_PATH = ROOT_DIR / "experiments/checkpoints/best_model_v3.pt"
print(f"📁 Checkpoint path: {CKPT_PATH}")
assert CKPT_PATH.exists(), f"❌ Checkpoint not found at: {CKPT_PATH}"


### 2. Load DINOv3 ViT-Small/16 Architecture & Checkpoint Weights


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("dinov3_vit", ROOT_DIR / "src/models/dinov3_vit.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

model = mod.build_dinov3_classifier(
    weights_path=None,
    num_classes=2,
    img_size=256,
    device=DEVICE
)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model.eval()

print("✅ Model loaded successfully!")
print(f"  • Architecture : Meta DINOv3 ViT-Small/16")
print(f"  • Total Params : {sum(p.numel() for p in model.parameters()):,}")
if "best_val_auc" in ckpt:
    print(f"  • Best Val AUC : {ckpt['best_val_auc']*100:.2f}%")


### 3. Single-Image Inference & Visualization Function


In [ ]:
# Image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_single_image(model, img_path):
    img_path = Path(img_path)
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found at: {img_path}")
    
    raw_img = Image.open(img_path).convert("RGB")
    tensor_img = transform(raw_img).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.bfloat16) if DEVICE == "cuda" else torch.no_grad():
            logits = model(tensor_img)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()[0]
    
    p_real = probs[0]
    p_fake = probs[1]
    pred_label = "FAKE (Deepfake)" if p_fake >= 0.5 else "REAL (Authentic)"
    confidence = max(p_real, p_fake) * 100
    theme_color = "#c0392b" if p_fake >= 0.5 else "#27ae60"
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={'width_ratios': [1, 1.2]})
    
    ax1.imshow(raw_img)
    ax1.set_title(f"Input: {img_path.name}", fontsize=11, fontweight="bold")
    ax1.axis("off")
    
    bars = ax2.barh(["Real (Authentic)", "Fake (Deepfake)"], [p_real * 100, p_fake * 100], color=["#27ae60", "#c0392b"], height=0.45, edgecolor="black", alpha=0.88)
    ax2.set_xlim(0, 105)
    ax2.set_xlabel("Probability (%)", fontsize=11, fontweight="bold")
    ax2.set_title(f"Prediction: {pred_label}\nConfidence: {confidence:.2f}%", fontsize=12, fontweight="bold", color=theme_color, pad=12)
    
    for bar in bars:
        w = bar.get_width()
        ax2.text(w + 2, bar.get_y() + bar.get_height() / 2, f"{w:.2f}%", va="center", ha="left", fontsize=10, fontweight="bold")
    
    ax2.grid(axis="x", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return {"prediction": pred_label, "p_real": float(p_real), "p_fake": float(p_fake), "confidence_percent": float(confidence)}


### 4. Interactive Test on Custom Image
Provide the absolute or relative path to any image below and execute the cell:


In [ ]:
# ============================================================
# Specify image path below:
# ============================================================
sample_img_path = ROOT_DIR / "data/test/Untitled.jpg"

if not sample_img_path.exists():
    # Fallback to test bundle image
    test_bundle_dir = Path("~/ Desktop/deepfake_test_suite_full_50k_exact_images/test_images")
    found_sample = list(test_bundle_dir.glob("**/*.png")) + list(test_bundle_dir.glob("**/*.jpg"))
    if found_sample:
        sample_img_path = found_sample[0]

print(f"🔍 Testing image: {sample_img_path}")
if sample_img_path and Path(sample_img_path).exists():
    result = predict_single_image(model, sample_img_path)
    print(f"📊 Result: {result['prediction']} with {result['confidence_percent']:.2f}% confidence")
else:
    print("⚠️ Please provide a valid image path to test.")
